In [ ]:
import sys
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix, RocCurveDisplay, 
    ConfusionMatrixDisplay
)

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

In [ ]:
DATA_PATH = Path("data/t2_transformed/merged_v1.csv")

In [ ]:
df = pd.read_csv(DATA_PATH)
df.shape

(91411, 115)

In [ ]:
drop_id_cols = ["MatchFk", "Patch"]
df = df.drop(columns=[c for c in drop_id_cols if c in df.columns], errors="ignore")

In [ ]:
df = df[df["BlueWin"] != df["RedWin"]].reset_index(drop=True)
df = df.drop(columns=["RedWin"], errors="ignore")

In [ ]:
lane_cols = [f"Lane_P{p}" for p in range(1, 11)]
for c in lane_cols:
    df[c] = df[c].replace("NONE", "JUNGLE")

In [ ]:
ohe_cols = df.select_dtypes(include="object").columns.tolist()

df = pd.get_dummies(df, columns=ohe_cols)

In [ ]:
# --- Team-level aggregates ---
blue_players = [f"P{i}" for i in range(1, 6)]
red_players  = [f"P{i}" for i in range(6, 11)]

for stat in ["TotalGold", "MinionsKilled", "kills", "deaths", "assists", "DmgDealt"]:
    blue_cols = [f"{stat}_{p}" for p in blue_players]
    red_cols  = [f"{stat}_{p}" for p in red_players]
    df[f"Blue_{stat}_sum"] = df[blue_cols].sum(axis=1)
    df[f"Red_{stat}_sum"]  = df[red_cols].sum(axis=1)
    df[f"Blue_{stat}_avg"] = df[blue_cols].mean(axis=1)
    df[f"Red_{stat}_avg"]  = df[red_cols].mean(axis=1)
    df[f"Diff_{stat}"]     = df[f"Blue_{stat}_sum"] - df[f"Red_{stat}_sum"]

# --- KDA ---
def safe_kda(k, d, a):
    return (k + a) / d.replace(0, 1)

df["Blue_KDA"] = safe_kda(df["Blue_kills_sum"], df["Blue_deaths_sum"], df["Blue_assists_sum"])
df["Red_KDA"]  = safe_kda(df["Red_kills_sum"],  df["Red_deaths_sum"],  df["Red_assists_sum"])
df["Diff_KDA"] = df["Blue_KDA"] - df["Red_KDA"]

# --- Objective differentials (already team-level) ---
for obj in ["BaronKills", "RiftHeraldKills", "DragonKills", "TowerKills"]:
    df[f"Diff_{obj}"] = df[f"Blue{obj}"] - df[f"Red{obj}"]

# Kill differential (already team-level)
df["Diff_Kills"] = df["BlueKills"] - df["RedKills"]

# --- Gold share (how evenly gold is distributed) ---
for side, players in [("Blue", blue_players), ("Red", red_players)]:
    gold_cols = [f"TotalGold_{p}" for p in players]
    df[f"{side}_GoldStd"] = df[gold_cols].std(axis=1)

In [ ]:
# Strategy A — "Aggregate-only" (team-level features, no per-player columns)
agg_feature_cols = [c for c in df.columns if c.startswith(("Blue_", "Red_", "Diff_"))]
agg_feature_cols += [f"Diff_{obj}" for obj in ["BaronKills", "RiftHeraldKills", "DragonKills", "TowerKills", "Kills"]]
# Also include objective raw counts
agg_feature_cols += ["BlueBaronKills", "BlueRiftHeraldKills", "BlueDragonKills", "BlueTowerKills", "BlueKills",
                     "RedBaronKills", "RedRiftHeraldKills", "RedDragonKills", "RedTowerKills", "RedKills"]
# Deduplicate
agg_feature_cols = sorted(set(agg_feature_cols))

In [ ]:
def prepare_data(df, feature_cols=None, target="BlueWin", val_size=0.15, test_size=0.15, random_state=1337, scale=False):

    if feature_cols is None:
        feature_cols = [c for c in df.columns if c != target]

    X = df[feature_cols].copy()
    y = df[target].copy()

    temp_size = val_size + test_size
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=temp_size, random_state=random_state, stratify=y
    )

    relative_test_size = test_size / temp_size
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=relative_test_size, random_state=random_state, stratify=y_temp
    )

    scaler = None
    X_train_sc, X_val_sc, X_test_sc = X_train, X_val, X_test

    if scale:
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_val_sc   = scaler.transform(X_val)
        X_test_sc  = scaler.transform(X_test)

    return {
        "X_train": X_train_sc,
        "X_val":   X_val_sc,
        "X_test":  X_test_sc,
        "y_train": y_train,
        "y_val":   y_val,
        "y_test":  y_test,
        "scaler":  scaler
    }

def unpack_data(data):
    return (
        data["X_train"],
        data["X_val"],
        data["X_test"],
        data["y_train"],
        data["y_val"],
        data["y_test"],
    )

In [ ]:
data_sc = prepare_data(df, feature_cols=agg_feature_cols, scale=True)
X_train_sc, X_val_sc, X_test_sc, y_train, y_val, y_test = unpack_data(data_sc)

data = prepare_data(df, feature_cols=agg_feature_cols)
X_train, X_val, X_test, y_train, y_val, y_test = unpack_data(data)

NameError: name 'df' is not defined

In [ ]:
def train_and_evaluate(name, model, data, results=None):
    if results is None:
        results = {}

    X_train, X_val, X_test, y_train, y_val, y_test = unpack_data(data)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    f1  = f1_score(y_test, y_pred)

    results[name] = {
        "model":  model,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "acc":    acc,
        "auc":    auc,
        "f1":     f1,
    }

    print(f" > {name}")
    print(f"   Accuracy: {acc:.4f}  |  AUC: {auc:.4f}  |  F1: {f1:.4f}")

    return results

In [ ]:
N_TRIALS = 50

def obj_logreg(trial):
    C = trial.suggest_float("C", 1e-4, 10, log=True)
    solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear", "saga"])
    penalty = "l2" if solver == "lbfgs" else trial.suggest_categorical("penalty", ["l1", "l2"])
    m = LogisticRegression(C=C, solver=solver, penalty=penalty, max_iter=2000, random_state=42)
    m.fit(X_train_sc, y_train)
    return roc_auc_score(y_val, m.predict_proba(X_val_sc)[:, 1])

def obj_dt(trial):
    m = DecisionTreeClassifier(
        max_depth=trial.suggest_int("max_depth", 2, 30),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 50),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 30),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        random_state=42,
    )
    m.fit(X_train.values, y_train.values)
    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])

def obj_rf(trial):
    m = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 500, step=50),
        max_depth=trial.suggest_int("max_depth", 4, 20),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 30),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        random_state=42, n_jobs=-1,
    )
    m.fit(X_train.values, y_train.values)
    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])

def obj_gb(trial):
    m = GradientBoostingClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 400, step=50),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 30),
        random_state=42,
    )
    m.fit(X_train.values, y_train.values)
    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])

def obj_xgb(trial):
    m = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 20),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
        tree_method="hist", eval_metric="logloss",
        early_stopping_rounds=30, random_state=42, n_jobs=-1, verbosity=0,
    )
    m.fit(X_train.values, y_train.values, eval_set=[(X_val.values, y_val.values)], verbose=False)
    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])

def obj_lgb(trial):
    m = lgb.LGBMClassifier(
        n_estimators=500,
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 50),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
        random_state=42, n_jobs=-1, verbose=-1,
    )
    m.fit(X_train.values, y_train.values, eval_set=[(X_val.values, y_val.values)],
          callbacks=[lgb.early_stopping(30, verbose=False)])
    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])

def obj_catboost(trial):
    m = CatBoostClassifier(
        iterations=500,
        depth=trial.suggest_int("depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1e-2, 10, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        random_seed=42, verbose=0, early_stopping_rounds=30,
    )
    m.fit(X_train.values, y_train.values, eval_set=(X_val.values, y_val.values))
    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])

def obj_mlp(trial):
    n1 = trial.suggest_int("n1", 32, 256, step=32)
    n2 = trial.suggest_int("n2", 16, 128, step=16)
    n3 = trial.suggest_int("n3", 8, 64, step=8)
    m = MLPClassifier(
        hidden_layer_sizes=(n1, n2, n3),
        activation=trial.suggest_categorical("activation", ["relu", "tanh"]),
        alpha=trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
        learning_rate_init=trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        batch_size=trial.suggest_categorical("batch_size", [128, 256, 512]),
        solver="adam", learning_rate="adaptive",
        max_iter=300, early_stopping=True, validation_fraction=0.15, random_state=42,
    )
    m.fit(X_train_sc, y_train)
    return roc_auc_score(y_val, m.predict_proba(X_val_sc)[:, 1])

In [ ]:
studies = {}
objectives = {
    "LogisticRegression": obj_logreg,
    "DecisionTree":       obj_dt,
    "RandomForest":       obj_rf,
    "GradientBoosting":   obj_gb,
    "XGBoost":            obj_xgb,
    "LightGBM":           obj_lgb,
    "CatBoost":           obj_catboost,
    "MLP":                obj_mlp,
}

for name, obj_fn in objectives.items():
    print(f"\n  > Tuning {name} ({N_TRIALS} trials) ...")
    study = optuna.create_study(direction="maximize", study_name=name)
    study.optimize(obj_fn, n_trials=N_TRIALS, show_progress_bar=False)
    studies[name] = study
    print(f"Best Val AUC: {study.best_value:.4f}")
    print(f"Best Params:  {study.best_params}")


  > Tuning LogisticRegression (50 trials) ...


In [ ]:
print("\n" + "=" * 60)
print("3. Retraining best models & evaluating on TEST set ...")
print("=" * 60)

def build_best_model(name, params):
    if name == "LogisticRegression":
        return LogisticRegression(**params, max_iter=2000, random_state=42)
    elif name == "DecisionTree":
        return DecisionTreeClassifier(**params, random_state=42)
    elif name == "RandomForest":
        return RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    elif name == "GradientBoosting":
        return GradientBoostingClassifier(**params, random_state=42)
    elif name == "XGBoost":
        return xgb.XGBClassifier(**params, n_estimators=500, tree_method="hist",
                                  eval_metric="logloss", early_stopping_rounds=30,
                                  random_state=42, n_jobs=-1, verbosity=0)
    elif name == "LightGBM":
        return lgb.LGBMClassifier(**params, n_estimators=500, random_state=42, n_jobs=-1, verbose=-1)
    elif name == "CatBoost":
        return CatBoostClassifier(**params, iterations=500, random_seed=42, verbose=0, early_stopping_rounds=30)
    elif name == "MLP":
        hl = (params.pop("n1"), params.pop("n2"), params.pop("n3"))
        lr = params.pop("lr")
        return MLPClassifier(hidden_layer_sizes=hl, learning_rate_init=lr, **params,
                             solver="adam", learning_rate="adaptive", max_iter=300,
                             early_stopping=True, validation_fraction=0.15, random_state=42)

results = {}
for name, study in studies.items():
    bp = study.best_params.copy()
    model = build_best_model(name, bp)

    # Choose scaled or unscaled data
    uses_scaled = name in ("LogisticRegression", "MLP")
    Xtr  = X_train_sc if uses_scaled else X_train.values
    Xv   = X_val_sc   if uses_scaled else X_val.values
    Xte  = X_test_sc  if uses_scaled else X_test.values

    # Fit with eval_set for boosting models
    if name == "XGBoost":
        model.fit(Xtr, y_train.values, eval_set=[(Xv, y_val.values)], verbose=False)
    elif name == "LightGBM":
        model.fit(Xtr, y_train.values, eval_set=[(Xv, y_val.values)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])
    elif name == "CatBoost":
        model.fit(Xtr, y_train.values, eval_set=(Xv, y_val.values))
    else:
        model.fit(Xtr, y_train.values)

    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    f1  = f1_score(y_test, y_pred)

    results[name] = {"model": model, "y_pred": y_pred, "y_prob": y_prob,
                     "acc": acc, "auc": auc, "f1": f1,
                     "best_params": study.best_params, "val_auc": study.best_value}

    print(f"\n  > {name}")
    print(f"    Val AUC:  {study.best_value:.4f}")
    print(f"    Test Acc: {acc:.4f} | Test AUC: {auc:.4f} | Test F1: {f1:.4f}")

In [ ]:
print("\n" + "=" * 60)
print("4. FINAL RESULTS (Test Set)")
print("=" * 60)

summary = pd.DataFrame({
    name: {"Val_AUC": r["val_auc"], "Test_Accuracy": r["acc"], "Test_AUC": r["auc"], "Test_F1": r["f1"]}
    for name, r in results.items()
}).T.sort_values("Test_AUC", ascending=False)

print("\n", summary.to_string(float_format="%.4f"))

# Classification reports for top 2
top2 = summary.index[:2].tolist()
for name in top2:
    print(f"\n{'-'*40}")
    print(f"Classification Report: {name}")
    print('-'*40)
    print(classification_report(y_test, results[name]["y_pred"], target_names=["Red Win", "Blue Win"]))

# Save results
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
summary.to_csv(OUTPUT_DIR / "optuna_tuning_results.csv")

# Save best params
params_df = pd.DataFrame({name: r["best_params"] for name, r in results.items()}).T
params_df.to_csv(OUTPUT_DIR / "best_hyperparameters.csv")

best_name = summary.index[0]
print(f"\n{'='*60}")
print(f"[BEST] {best_name}")
print(f"  Test Accuracy: {results[best_name]['acc']:.4f}")
print(f"  Test AUC-ROC:  {results[best_name]['auc']:.4f}")
print(f"  Test F1-Score:  {results[best_name]['f1']:.4f}")
print(f"  Params: {results[best_name]['best_params']}")
print(f"\nResults saved to: {OUTPUT_DIR.resolve()}")
print("=" * 60)